In [ ]:
# get the code
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# use seaborn plotting defaults
import seaborn as sns; sns.set()

# import package functions (main: sys.path.append('../code') and the modules in code/)
import mycnv
from mycnv.plot import plot_cov
from mycnv.rolling_cov import rolling_coverage
from mycnv.combine import filter_cov

# paths and parameters: ../config/config_dev.yaml via dev_config.py (main: hard-coded Dropbox paths)
from dev_config import load_dev
dev = load_dev()
plot_path = dev["plot_path"]
fig_path = dev["fig_path"]
print(f"mycnv {mycnv.__version__} ({os.path.dirname(mycnv.__file__)})")
print(f"sample {dev['sample']} ({dev['study']}) -> {dev['out_dir']}")

### get the config
+ `dev_config.load_dev()` reads `config/config_dev.yaml`: the data paths of the local cache and the Gen-2 CNV params
  (`mycnv.config.load_config`, deep-merged over the mycnv defaults = the somVarWES study config)
+ headless runs: `MYCNV_NB_SAMPLE=<key> jupyter nbconvert --execute ...` (keys under `samples:`)

In [ ]:
CNVconfig = dev["CNVconfig"]

### load coverage data and visualize

In [ ]:
sample = dev["sample"]
# the study's raw/{sample}.cov.gz = output of notebook 2 (main: output/cov/{sample}.cov.gz)
cov_rawdf = pd.read_csv(dev["raw_cov"], sep="\t")
cov_rawdf[:5]

In [ ]:
fig_params = dict(
    figsize=(42,6),
    colormap='coolwarm_r',
    color_chroms=True,
    ylim=(-2,3)
)
log2 = dict(
        title='log2ratio',
        plot_type='scatter',   # ['line', 'scatter']
        data='log2ratio2',
        plot_args=dict(
            linewidth=0.3,
            color='black',
            s=.5,
            alpha=.7
        ))


fig, ax, _, chrom_df = plot_cov(cov_rawdf, plots=[log2], chroms='all', region='', **fig_params)
# fig.savefig(os.path.join(cnvdata, "figures/03_GC_cov.jpeg"))
# cov_noGC_df = pd.read_csv(os.path.join(output_path, f"cov/{sample}.noGC.cov.gz"), sep="\t", compression="gzip")
# fig.savefig(os.path.join(cnvdata, "figures/03_GC_cov.jpeg"))

## filter coverage data

In [ ]:
CNVconfig = dev["CNVconfig"]

cov_df = filter_cov(cov_rawdf, config=CNVconfig)

fig, ax, _, chrom_df = plot_cov(cov_df, plots=[log2], chroms='all', region='', **fig_params)

## run the code

In [ ]:
from mycnv.rolling_cov import rolling_coverage
roll_df = rolling_coverage(cov_df, config=CNVconfig)

log2mean = dict(
        title='rollinglog2ratio',
        plot_type='line',   # ['line', 'ascatter']
        data='log2ratio2_mean',
        plot_args=dict(
            linewidth=1,
            color='yellow',
            alpha=.7
        )
    )

# main named these plot dicts llh/llhdiff; the step-by-step cells rebind llh to the function (TypeError in
# the region plot below on a fresh run), so they are llh_plot/llhdiff_plot here
llh_plot = dict(
        title='llh',
        plot_type='line',   # ['line', 'scatter']
        data='covLLH2_sum',
        plot_args=dict(
            linewidth=1,
            color='blue',
            alpha=.7
        )
    )

llhdiff_plot = dict(
        title='llh',
        plot_type='line',   # ['line', 'scatter']
        data='covLLH2_sumDiff',
        plot_args=dict(
            linewidth=1,
            color='green',
            alpha=.7
        )
    )

log2L = dict(
        title='rollinglog2ratio',
        plot_type='line',   # ['line', 'scatter']
        data='log2ratio2_meanL',
        plot_args=dict(
            linewidth=1,
            color='white',
            alpha=.7
        ))

log2R = dict(
        title='rollinglog2ratio',
        plot_type='line',   # ['line', 'scatter']
        data='log2ratio2_meanR',
        plot_args=dict(
            linewidth=1,
            color='black',
            alpha=.7
        ))
r1 = 'chr20:47Mb-57Mb'
r2 = 'chr4:110Mb-140Mb'
fig, _, _, _ = plot_cov(roll_df, plots=[log2,log2mean, llh_plot, llhdiff_plot], chroms='all', region=r1, **fig_params)

In [ ]:
roll_df.to_csv(dev["roll_cov"], sep="\t", index=False, compression="gzip")

## Step by step
+ make a copy for computation

In [ ]:
df = cov_df.loc[:,['Chr', 'Pos', 'ExonPos', 'FullExonPos', 'Cov1', 'log2ratio1', 'Cov2', 'log2ratio2']].copy()
df[:5]

### assumption: 
+ log2ratio is normal-distributed around mean value:
+ log-likelihood should be far below average at CNV areas
### compute the llh for center mass for entire sample
+ get the global mean
+ compute loglikelihood
+ rolling sum for chromosome

In [ ]:
import numpy.random as rnd

col = "log2ratio1"
sigma = df.query(f'-0.5 < {col} < .5')[col].std() * .9
# get the mean of the center band
mean = df.query(f'-0.5 < {col} < .5')[col].mean()
print("sigma:", sigma)
print("mean:", mean)
def llh(data, mean, sigma):
    s = np.sqrt(2 * np.pi) * sigma
    return np.exp((data - mean)**2 / (-2*(sigma**2))) / s

r = np.linspace(-2,2,10000)
fig1, ax = plt.subplots(figsize=(10,4))
_ = ax.scatter(r, llh(r, mean, sigma), s=.5, alpha=0.5)
_ = ax.scatter(df[col], rnd.random(len(df.index))*4, s=.01, alpha=.1)
_ = ax.set_xlim(-1.5,2)
fig1.savefig(os.path.join(plot_path, f"{sample}.log2gauss.jpg"))

In [ ]:
# canonical code (was an inline copy of code/rollingCov.py): mycnv.rolling.llh, mycnv.rolling_cov.compute_covLLH[_col]
from mycnv.rolling import llh
from mycnv.rolling_cov import compute_covLLH_col, compute_covLLH

In [ ]:
llh_df = compute_covLLH(df, config=CNVconfig)
llh_df

## rolling window for log2ratio and llh
+ compute the mean for log2ratio

In [ ]:
# canonical code (was an inline copy of code/rollingCNV.py): mycnv.rolling
# changes vs main (CHANGELOG.md): windows per chromosome, diff_exp is read, ddof only for std/var
from mycnv.rolling import interpolate, normalize_df, one_col_rolling

In [ ]:
# canonical code (was an inline copy of code/rollingCNV.py:rolling_data and code/rollingCov.py:rolling_coverage)
from mycnv.rolling import rolling_data
from mycnv.rolling_cov import rolling_coverage

In [ ]:
# main dropped L/R columns left over from an interactive session (KeyError on a fresh run); mycnv does not add them
df = df.drop(columns=['L', 'R'], errors='ignore')
CNVconfig = dev["CNVconfig"]

roll_df = rolling_coverage(df, config=CNVconfig)
roll_df[:10]

### visualize

In [ ]:
fig, _, _, _ = plot_cov(roll_df, plots=[log2,log2mean], chroms='all', region='', **fig_params)

In [ ]:
r1 = 'chr20:47Mb-57Mb'
r2 = 'chr4:110Mb-140Mb'
fig, _, _, _ = plot_cov(roll_df, plots=[log2,log2mean, llh_plot, llhdiff_plot], chroms='all', region=r1, **fig_params)

In [ ]:


# chroms = ['chr3', 'chr4', 'chr5', 'chr6', 'chr8', 'chr9', 'chr20']

fig, _, _, _ = plot_cov(roll_df, plots=[log2,log2mean], chroms='all', region='', **fig_params)